# Socioeconomic Risk Metrics

This notebook creates sub-national socioeconomic risk metrics for the national tool.

It is designed to grow section by section. Each hazard section creates a tidy metrics table keyed by `adm_id`; the final section exports one CSV per hazard. The current flooding section summarizes expected annual flooded population metrics by demographic and wealth group.


## 0. Setup

Set the country, administrative level, hazard runs, and input/output paths here. The current socioeconomic flooding layer is already aggregated to ADM1 regions, so this notebook reads the GeoPackage attribute table and reshapes the expected annual rows into dashboard-ready metrics.


In [ ]:
from pathlib import Path
import re
import sqlite3

import pandas as pd

pd.set_option("display.max_columns", 100)


In [ ]:
# Core parameters
COUNTRY_ISO3 = "KEN"
COUNTRY_NAME = "Kenya"
ADMIN_LEVEL = "adm1"  # options currently available: adm0, adm1, adm2
MODULE = "socioeconomic"
HAZARD = "flooding"
MODEL_RUN = "river-jrc_population_risk_metrics"

# Resolve paths whether the notebook is run from the repo root or notebooks/.
NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = NOTEBOOK_CWD.parent if NOTEBOOK_CWD.name == "notebooks" else NOTEBOOK_CWD

RAW_SOCIOECONOMIC_DIR = REPO_ROOT / "data" / "raw" / COUNTRY_ISO3 / MODULE
OUTPUT_DIR = REPO_ROOT / "results" / COUNTRY_ISO3 / MODULE / HAZARD
OUTPUT_PATH = OUTPUT_DIR / f"{COUNTRY_ISO3}_{ADMIN_LEVEL}_{MODULE}_{HAZARD}_metrics.csv"

POPULATION_RISK_PATH = RAW_SOCIOECONOMIC_DIR / HAZARD / "KEN_ADM1_jrc_population_risk_metrics.gpkg"
POPULATION_RISK_LAYER = "KEN_ADM1_jrc_population_risk_metrics"

# Keep only expected annual risk maps; return-period rows such as RP10/RP100 are intentionally excluded.
EXPECTED_ANNUAL_RISK_MAPS = ["AAR", "AAR_protected"]
RISK_MAP_METRIC_PREFIX = {
    "AAR": "flooded_pop_ea",
    "AAR_protected": "flooded_pop_ea_protected",
}

POPULATION_GROUP_TOKENS = {
    "total": "total",
    "female": "female",
    "male": "male",
    "children_under5": "under_5",
    "school_age_5_14": "school_children_5_14",
    "working_age_15_64": "working_age_15_64",
    "female_15_49": "female_childbearing_15_49",
    "older_65plus": "older_65_plus",
    "wealth_q1": "wealth_q1",
    "wealth_q2": "wealth_q2",
    "wealth_q3": "wealth_q3",
    "wealth_q4": "wealth_q4",
    "wealth_q5": "wealth_q5",
}

POPULATION_RISK_PATH, OUTPUT_PATH


## 0.1 Helper Functions

The source GeoPackage stores one row for each administrative region, risk map, and population group. The helper functions below read only the attribute columns, validate the expected schema, and pivot the expected annual rows into one record per administrative region.


In [ ]:
def check_path_exists(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")


def quote_identifier(identifier: str) -> str:
    return '"' + identifier.replace('"', '""') + '"'


def normalize_metric_token(value: object) -> str:
    """Convert a category value into a compact metric-name token."""
    if value in POPULATION_GROUP_TOKENS:
        return POPULATION_GROUP_TOKENS[value]
    token = str(value).lower().strip()
    token = re.sub(r"[^a-z0-9]+", "_", token).strip("_")
    return token or "unknown"


def validate_columns(frame: pd.DataFrame, required_columns: set[str], label: str) -> None:
    missing_columns = required_columns.difference(frame.columns)
    if missing_columns:
        raise ValueError(f"{label} is missing required columns: {sorted(missing_columns)}")


def read_gpkg_attributes(gpkg_path: Path, layer_name: str) -> pd.DataFrame:
    """Read a GeoPackage layer's non-geometry attributes with sqlite3."""
    check_path_exists(gpkg_path, "Population risk GeoPackage")
    with sqlite3.connect(gpkg_path) as connection:
        geometry_columns = pd.read_sql_query(
            """
            SELECT column_name
            FROM gpkg_geometry_columns
            WHERE table_name = ?
            """,
            connection,
            params=[layer_name],
        )["column_name"].tolist()

        column_info = pd.read_sql_query(f"PRAGMA table_info({quote_identifier(layer_name)})", connection)
        if column_info.empty:
            raise ValueError(f"Layer not found in GeoPackage: {layer_name}")

        attribute_columns = [
            column
            for column in column_info["name"].tolist()
            if column not in geometry_columns and column != "fid"
        ]
        select_columns = ", ".join(quote_identifier(column) for column in attribute_columns)
        return pd.read_sql_query(f"SELECT {select_columns} FROM {quote_identifier(layer_name)}", connection)


def population_risk_metrics_by_admin(
    risk_table: pd.DataFrame,
    expected_annual_risk_maps: list[str],
    risk_map_metric_prefix: dict[str, str],
) -> pd.DataFrame:
    """Pivot expected annual flooded population counts to one row per admin region."""
    required_columns = {
        "shapeID",
        "shapeName",
        "ISO3",
        "admin_level",
        "risk_map",
        "population_group",
        "population_group_type",
        "exposed_population",
    }
    validate_columns(risk_table, required_columns, "Population risk table")

    metrics_source = risk_table[risk_table["risk_map"].isin(expected_annual_risk_maps)].copy()
    if metrics_source.empty:
        raise ValueError(f"No expected annual risk-map rows found for: {expected_annual_risk_maps}")

    missing_risk_maps = sorted(set(expected_annual_risk_maps).difference(metrics_source["risk_map"].unique()))
    if missing_risk_maps:
        raise ValueError(f"Expected annual risk maps missing from source table: {missing_risk_maps}")

    duplicate_keys = metrics_source.duplicated(["shapeID", "risk_map", "population_group"])
    if duplicate_keys.any():
        duplicates = metrics_source.loc[duplicate_keys, ["shapeID", "risk_map", "population_group"]]
        raise ValueError(f"Duplicate admin/risk-map/population-group rows found. Example:\n{duplicates.head()}")

    admin_identifiers = (
        metrics_source[["shapeID", "shapeName", "ISO3", "admin_level"]]
        .drop_duplicates("shapeID")
        .rename(
            columns={
                "shapeID": "adm_id",
                "shapeName": "adm_name",
                "ISO3": "country_iso3",
            }
        )
        .sort_values("adm_name")
        .reset_index(drop=True)
    )
    admin_identifiers["country_name"] = COUNTRY_NAME
    admin_identifiers["admin_level"] = admin_identifiers["admin_level"].str.upper()
    admin_identifiers["module"] = MODULE
    admin_identifiers["hazard"] = HAZARD
    admin_identifiers["model_run"] = MODEL_RUN

    standard_columns = [
        "country_iso3",
        "country_name",
        "admin_level",
        "adm_id",
        "adm_name",
        "module",
        "hazard",
        "model_run",
    ]
    hazard_metrics = admin_identifiers[standard_columns].copy()

    for risk_map in expected_annual_risk_maps:
        metric_prefix = risk_map_metric_prefix[risk_map]
        risk_map_rows = metrics_source[metrics_source["risk_map"] == risk_map].copy()
        risk_map_rows["population_group_token"] = risk_map_rows["population_group"].map(normalize_metric_token)
        risk_map_rows["exposed_population"] = pd.to_numeric(risk_map_rows["exposed_population"], errors="coerce").fillna(0)

        exposed_population = risk_map_rows.pivot(
            index="shapeID",
            columns="population_group_token",
            values="exposed_population",
        )
        exposed_population = exposed_population.rename(columns=lambda token: f"{metric_prefix}_{token}")
        exposed_population.columns.name = None

        risk_map_metrics = exposed_population.reset_index()
        risk_map_metrics = risk_map_metrics.rename(columns={"shapeID": "adm_id"})
        hazard_metrics = hazard_metrics.merge(risk_map_metrics, on="adm_id", how="left", validate="one_to_one")

    count_columns = [column for column in hazard_metrics.columns if column.startswith("flooded_pop_ea")]
    hazard_metrics[count_columns] = hazard_metrics[count_columns].fillna(0).round(3)
    return hazard_metrics


## 1. Flooding Expected Annual Population Exposure

This section summarizes expected annual flooded population metrics from the JRC population risk layer. Return-period maps (`RP10`, `RP20`, `RP50`, etc.) are intentionally excluded.

Metrics being reported for each demographic and wealth group:

- `flooded_pop_ea_<population_group>`: expected annual flooded population from `risk_map = AAR`.
- `flooded_pop_ea_protected_<population_group>`: protected expected annual flooded population from `risk_map = AAR_protected`.


In [ ]:
population_risk_table = read_gpkg_attributes(POPULATION_RISK_PATH, POPULATION_RISK_LAYER)

available_risk_maps = sorted(population_risk_table["risk_map"].dropna().unique().tolist())
available_population_groups = sorted(population_risk_table["population_group"].dropna().unique().tolist())

available_risk_maps, available_population_groups


In [ ]:
flooding_metrics = population_risk_metrics_by_admin(
    population_risk_table,
    expected_annual_risk_maps=EXPECTED_ANNUAL_RISK_MAPS,
    risk_map_metric_prefix=RISK_MAP_METRIC_PREFIX,
)

flooding_metrics.head()


## 2. Export Socioeconomic Metrics

The output uses the same standard identifier columns as the other module notebooks, followed by the metric columns created above.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
flooding_metrics.to_csv(OUTPUT_PATH, index=False)

print(f"Exported {len(flooding_metrics):,} {HAZARD} rows and {len(flooding_metrics.columns):,} columns to:")
print(OUTPUT_PATH)
